# LifeMate private inference
The notebook uses the same offline entry point as the Docker script. The CLI initializes seeds and models once and measures each sample from audio loading to official output.

## Fixed seed and offline configuration
The shared CLI seeds Python random, NumPy and PyTorch before loading models. The child interpreter also starts with a fixed Python hash seed.

In [ ]:
import os
from pathlib import Path

env = os.environ.copy()
env.setdefault('LIFEMATE_PLANNER_MODE', 'native')
env.setdefault('LIFEMATE_PLANNER_MODEL_DIR', '/models/planner')
env['PYTHONHASHSEED'] = '20261112'
env['HF_HUB_OFFLINE'] = '1'
env['TRANSFORMERS_OFFLINE'] = '1'


## Mounted input
Only released metadata and the mounted package entry point are checked directly; ToolEnv owns runtime state access.

In [ ]:
input_split = Path('/data/private_test')
for relative in ('private_test.jsonl', 'private_test_tools_catalog.json', 'zaic_tools/__init__.py'):
    assert (input_split / relative).is_file(), f'Missing released input: {relative}'


## Local model loading
The inference cell loads both models once through the shared CLI after initializing seeds. Confirm the local model directories here; no model or dataset download is performed.

In [ ]:
planner_dir = Path(env['LIFEMATE_PLANNER_MODEL_DIR'])
asr_dir = Path('/models/asr')
assert planner_dir.is_dir(), f'Missing local planner: {planner_dir}'
assert asr_dir.is_dir(), f'Missing local ASR: {asr_dir}'
print(f'Planner: {planner_dir}; ASR: {asr_dir}; mode: {env["LIFEMATE_PLANNER_MODE"]}')


## Model load and inference
Run the same entry point as predict.sh. Output predictions use the same seeds, models, limits and mounted ToolEnv as the script; timing values are measured afresh.

In [ ]:
import subprocess

subprocess.run(['bash', '/code/predict.sh', '/result/jupyter_submission.jsonl'], check=True, env=env)

In [ ]:
import json
from pathlib import Path

def read_predictions(path):
    with path.open(encoding='utf-8') as source:
        records = [json.loads(line) for line in source]
    assert all(type(row['time']) is int and 0 <= row['time'] <= 30000 for row in records)
    return [(row['id'], row['tool_calls'], row['state_hash']) for row in records]

notebook_records = read_predictions(Path('/result/jupyter_submission.jsonl'))
assert notebook_records
script_output = Path('/result/submission.jsonl')
if script_output.exists():
    assert notebook_records == read_predictions(script_output)
print(f'{len(notebook_records)} official records')